# Weighted Time-Synchronous Averaging (WTSA) — PPG Frame Export

This notebook exports averaged PPG frames using **Weighted Time-Synchronous Averaging (WTSA)**
in place of the simple arithmetic mean used in `lambert_same_csv_frames.ipynb`.

Two weighted averaging methods from Momot (2009) are implemented:

- **WAPM** (Weighted Averaging via Partition of the input set in the tiMe domain) —  
  criterion-function minimization over `C` interlaced disjoint subsets of segments.
- **EBWA-C** (Empirical Bayesian Weighted Averaging — Cauchy prior, p=½) —  
  Bayesian EM approach; consistently best accuracy in the original paper.

## Output files

| Flag | File | Schema |
|---|---|---|
| `SAVE_SEGMENTS` | `ppg_frames_wtsa.csv` | One row per waveform sample (matches `ppg_frames.csv`) |
| `SAVE_WITH_EPR` | `ppg_frames_wtsa_with_epr.csv` | Segment rows + EPR columns appended |
| `SAVE_EPR_ONLY` | `ppg_frames_wtsa_epr_1hz.csv` | **One row per second** (1 Hz), EPR + ABG/FHR labels — no waveform samples |

The EPR-only CSV has ABG/FHR labels joined directly (no separate `randall_concat` run needed).  
The `seconds` column is the actual elapsed recording second, so it aligns correctly with  
`fetal1_combined.csv` even when some windows have no valid segments.


In [1]:
import sys
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, argrelextrema

workspace_root = Path.cwd().parent if Path.cwd().name == "visualizations" else Path.cwd()
lambert_dir = workspace_root / "lambert"
if str(lambert_dir) not in sys.path:
    sys.path.append(str(lambert_dir))

import FrequencyWarping as fw
import PSA as mypsa

if str(workspace_root / "src") not in sys.path:
    sys.path.append(str(workspace_root / "src"))

from psafe_custom.apply_pasfe import get_sheep_dataframe, get_sheep_experiment_id, get_sorted_ac_columns
from psafe_custom.randall.epr_export import EPR_VALUE_HEADERS, experiment_id_for_sheep_index


In [ ]:
# ─── Output control ────────────────────────────────────────────────────────
SAVE_SEGMENTS = False   # ppg_frames_wtsa.csv          — one row per waveform sample
SAVE_WITH_EPR = False   # ppg_frames_wtsa_with_epr.csv — segment rows + EPR columns appended
SAVE_EPR_ONLY = True   # ppg_frames_wtsa_epr_1hz.csv  — one row per second, EPR + ABG/FHR labels

# ─── Averaging method ──────────────────────────────────────────────────────
# "EBWA_C" — empirical Bayesian EM with Cauchy prior (p=1/2); best overall accuracy (Momot 2009)
# "WAPM"   — criterion-function minimization over C interlaced subsets (2–4 recommended)
AVERAGING_METHOD = "EBWA_C"
WAPM_C           = 3    # Not used for EBWA_C, but recommended 2–4 for WAPM (Momot 2009)
EBWA_EPSILON     = 1e-6 # Recommended 1e-6, could try smaller for more precision at the cost of runtime (Momot 2009) Maybe range of 1e-4 to 1e-8?

# ─── Pipeline constants ────────────────────────────────────────────────────
FRAMES_TO_AVERAGE       = 120
FS                      = 80
HP_CUTOFF_HZ            = 1.0
HP_ORDER                = 4
FHR_MA_WINDOW_SECONDS   = 60
PEAK_DISTANCE_HINT      = 10
SEGMENT_TOLERANCE_RATIO = 0.2
DIA_FLOOR               = 0.0   # EPR denominator floor (matches lambert_same_frames.ipynb)

# ─── Paths ─────────────────────────────────────────────────────────────────
_OUT_DIR = Path("/home/rlfowler/Documents/research/psafe_custom/data")
OUT_CSV_SEGMENTS  = _OUT_DIR / "ppg_frames_wtsa.csv"
OUT_CSV_WITH_EPR  = _OUT_DIR / "ppg_frames_wtsa_with_epr.csv"
# OUT_CSV_EPR_ONLY  = _OUT_DIR / "wtsa_epr.csv" # Name for EBWA_C output
OUT_CSV_EPR_ONLY  = _OUT_DIR / f"wapm_epr_{WAPM_C}.csv" # Name for WAPM output

# fetal1_combined.csv: source of ABG/FHR labels, joined on experiment_id + seconds
ABG_SOURCE_CSV_PATH    = Path("/home/rlfowler/Documents/research/TFOEstimation/data/dataset/fetal/fetal1_combined.csv")
NORM_EXPERIMENT_ID_COL = "_experiment_id_norm"

# Columns to carry into both the segment CSV and the EPR-only CSV
ABG_COLUMNS = [
    "Fetal ABG pH",
    "Fetal ABG sO2 (%)",
    "Fetal ABG pO2 (mmHg)",
    "Fetal ABG pCO2 (mmHg)",
    "Fetal ABG BP",
    "Fetal ABG Lac (mmol/L)",
    "FHR_Clean",
    "FHR",
    "fSaO2",
    "Fetal ABG HCO3 (mmol/L)",
    "Fetal ABG BE (mmol/L)",
    "Fetal ABG CBE (mmol/L)",
    "Fetal ABG SBE (mmol/L)",
    "Fetal ABG pH Change",
    "Fetal ABG sO2 (%) Change",
    "Fetal ABG pO2 (mmHg) Change",
    "Fetal ABG pCO2 (mmHg) Change",
    "Fetal ABG BP Change",
    "Fetal ABG Lac (mmol/L) Change",
    "FHR_Clean Change",
    "FHR Change",
    "fSaO2 Change",
    "Fetal ABG HCO3 (mmol/L) Change",
    "Fetal ABG BE (mmol/L) Change",
    "Fetal ABG CBE (mmol/L) Change",
    "Fetal ABG SBE (mmol/L) Change",
]

# Ordered channel columns (WL1 then WL2, matching epr_export convention)
ORDERED_SIGNAL_COLS = [f"ch{i+1}voltsWL1" for i in range(5)] + [f"ch{i+1}voltsWL2" for i in range(5)]


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
#  WEIGHTED AVERAGING IMPLEMENTATIONS  (Momot, Biomed. Signal Proc. 2009)
# ═══════════════════════════════════════════════════════════════════════════

def _wapm_weights_two_subsets(Y1: np.ndarray, Y2: np.ndarray, epsilon: float = 1e-6):
    """
    WAPM weight vectors for two disjoint segment subsets.
    Y1, Y2 are (N_samples x M_segments) matrices.
    Implements equations (22)-(26) from Momot (2009) via Picard iteration.
    """
    ones1 = np.ones((Y1.shape[1], 1))
    ones2 = np.ones((Y2.shape[1], 1))
    Y1tY1_inv = np.linalg.pinv(Y1.T @ Y1)
    Y2tY2_inv = np.linalg.pinv(Y2.T @ Y2)
    Y1tY2 = Y1.T @ Y2
    Y2tY1 = Y2.T @ Y1
    denom1 = float(ones1.T @ Y1tY1_inv @ ones1)
    denom2 = float(ones2.T @ Y2tY2_inv @ ones2)
    w2 = np.ones((Y2.shape[1], 1)) / Y2.shape[1]
    for _ in range(1000):
        w2_prev = w2.copy()
        tmp1 = Y1tY1_inv @ Y1tY2 @ w2
        lam1 = (1.0 - float(ones1.T @ tmp1)) / denom1
        w1 = tmp1 + lam1 * (Y1tY1_inv @ ones1)
        tmp2 = Y2tY2_inv @ Y2tY1 @ w1
        lam2 = (1.0 - float(ones2.T @ tmp2)) / denom2
        w2 = tmp2 + lam2 * (Y2tY2_inv @ ones2)
        if np.linalg.norm(w2 - w2_prev) < epsilon:
            break
    return w1.flatten(), w2.flatten()


def wapm_average(segments: np.ndarray, C: int = 3, epsilon: float = 1e-6) -> np.ndarray:
    """
    WAPM: Weighted Averaging via Partition of the input set in the tiMe domain.

    segments : (M, N) array — M segments each resampled to N samples, single channel.
    C        : number of interlaced disjoint subsets (2-4 recommended).

    Interlaced partitioning: subset c contains segments at indices c, c+C, c+2C, ...
    as reported best-performing in Momot (2009).
    """
    M, N = segments.shape
    if M == 1:
        return segments[0].copy()
    C = min(C, M)
    subsets = [segments[c::C] for c in range(C)]
    if C == 2:
        Y1, Y2 = subsets[0].T, subsets[1].T
        w1, w2 = _wapm_weights_two_subsets(Y1, Y2, epsilon)
        return (len(w1) * (Y1 @ w1) + len(w2) * (Y2 @ w2)) / M
    weighted_sum = np.zeros(N)
    total_weight = 0.0
    prev_subset = subsets[0]
    for c in range(1, C):
        Y1, Y2 = prev_subset.T, subsets[c].T
        w1, w2 = _wapm_weights_two_subsets(Y1, Y2, epsilon)
        M1, M2 = len(w1), len(w2)
        weighted_sum += M1 * (Y1 @ w1) + M2 * (Y2 @ w2)
        total_weight += M1 + M2
        prev_subset = ((Y1 @ w1 + Y2 @ w2) / 2.0).reshape(1, N)
    return weighted_sum / total_weight if total_weight > 0 else np.mean(segments, axis=0)


def ebwa_c_average(segments: np.ndarray, epsilon: float = 1e-6) -> np.ndarray:
    """
    EBWA-C: Empirical Bayesian Weighted Averaging with Cauchy prior (p=1/2).

    Implements Section 4.2 of Momot (2009). Hyperparameter lambda is estimated
    from the sample interquartile range of the current signal estimate (eq. 45).

    segments : (M, N) array — M segments each resampled to N samples, single channel.
    """
    M, N = segments.shape
    if M == 1:
        return segments[0].copy()
    v = np.mean(segments, axis=0)
    for _ in range(500):
        v_prev = v.copy()
        residuals_sq = np.maximum(np.sum((segments - v[None, :]) ** 2, axis=1), 1e-12)
        alpha = N / residuals_sq                                        # eq. 36 — (M,)
        q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
        lam = max(((q3 - q1) ** 2) / 8.0, 1e-12)                      # eq. 45
        beta = 2.0 / (v ** 2 + 2.0 * lam)                             # eq. 37 — (N,), p=1/2 → 2p+1=2
        v = (alpha @ segments) / (beta + np.sum(alpha))                # eq. 35
        if np.linalg.norm(v - v_prev) < epsilon:
            break
    return v


def weighted_average_segments(segments: np.ndarray, method: str = "EBWA_C",
                               wapm_c: int = 3, epsilon: float = 1e-6) -> np.ndarray:
    """Dispatch to the selected weighted averaging method for a single channel."""
    if method == "WAPM":
        return wapm_average(segments, C=wapm_c, epsilon=epsilon)
    elif method == "EBWA_C":
        return ebwa_c_average(segments, epsilon=epsilon)
    else:
        raise ValueError(f"Unknown averaging method: {method!r}. Choose 'WAPM' or 'EBWA_C'.")


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
#  PIPELINE HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def normalize_experiment_id(value):
    text = "" if pd.isna(value) else str(value).strip()
    if text.lower().endswith(".csv"):
        text = text[:-4]
    return text.lower()

def ensure_csv_suffix(value):
    text = "" if pd.isna(value) else str(value).strip()
    return text if text.lower().endswith(".csv") else f"{text}.csv"

def _load_abg_source() -> pd.DataFrame:
    """Load fetal1_combined.csv and normalise experiment IDs for joining."""
    src = pd.read_csv(ABG_SOURCE_CSV_PATH)
    src[NORM_EXPERIMENT_ID_COL] = src["experiment_id"].map(normalize_experiment_id)
    src["seconds"] = pd.to_numeric(src["seconds"], errors="coerce")
    keep_cols = [NORM_EXPERIMENT_ID_COL, "seconds"] + [c for c in ABG_COLUMNS if c in src.columns]
    return src[keep_cols].copy()

def _interp_abg_at_times(abg_source: pd.DataFrame, experiment_id: str,
                          times: np.ndarray) -> pd.DataFrame:
    """
    Interpolate all ABG/FHR columns from abg_source at the given times (seconds).
    Returns a DataFrame with one row per entry in times.
    """
    norm_id = normalize_experiment_id(experiment_id)
    exp_src = abg_source[abg_source[NORM_EXPERIMENT_ID_COL] == norm_id].copy()
    out = pd.DataFrame(index=np.arange(len(times)))
    if exp_src.empty:
        for col in ABG_COLUMNS:
            out[col] = np.nan
        return out
    sec = pd.to_numeric(exp_src["seconds"], errors="coerce").to_numpy(dtype=float)
    for col in ABG_COLUMNS:
        if col not in exp_src.columns:
            out[col] = np.nan
            continue
        vals = pd.to_numeric(exp_src[col], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(sec) & np.isfinite(vals)
        if finite.sum() == 0:
            out[col] = np.nan
            continue
        t_f, v_f = sec[finite], vals[finite]
        order = np.argsort(t_f)
        out[col] = np.interp(times, t_f[order], v_f[order], left=np.nan, right=np.nan)
    return out

def _accepted_segment_meta(markers, fhr_hz, tolerance_ratio=SEGMENT_TOLERANCE_RATIO):
    markers = np.asarray(markers, dtype=int)
    fhr_hz = np.asarray(fhr_hz, dtype=float).reshape(-1)
    starts, raw_lens = [], []
    if len(markers) < 2:
        return starts, raw_lens
    for i in range(len(markers) - 1):
        s, e = int(markers[i]), int(markers[i + 1])
        idxlen = e - s
        if idxlen <= 1:
            continue
        tidx = int(np.clip(int(np.floor((s + e) / (2.0 * FS))), 0, len(fhr_hz) - 1))
        expected = FS / max(float(fhr_hz[tidx]), 1e-6)
        tol = tolerance_ratio * expected
        if (idxlen > expected - tol) and (idxlen < expected + tol):
            starts.append(s)
            raw_lens.append(float(idxlen))
    return starts, raw_lens

def highpass_filter(data, fs, cutoff=HP_CUTOFF_HZ, order=HP_ORDER):
    b, a = butter(order, cutoff / (0.5 * fs), btype="high", analog=False)
    return filtfilt(b, a, data, axis=0)

def _trim_to_common_length(ppg, fhr_1hz, fsao2_1hz, fs=FS):
    trim_sec = min(ppg.shape[0] // fs, len(fhr_1hz), len(fsao2_1hz))
    return ppg[:fs * trim_sec, :], fhr_1hz.iloc[:trim_sec, :], fsao2_1hz.iloc[:trim_sec], trim_sec

def epr_norm_ma(ac_hp: float, dia_raw: float, dia_floor: float = DIA_FLOOR) -> float:
    """Modulation ratio: (AC + DC) / DC, matching lambert_same_frames.ipynb."""
    if not (np.isfinite(ac_hp) and np.isfinite(dia_raw)):
        return float("nan")
    if dia_raw <= dia_floor:
        return float("nan")
    return (ac_hp + dia_raw) / dia_raw


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
#  WTSA PER-SHEEP COMPUTATION
# ═══════════════════════════════════════════════════════════════════════════

def compute_wtsa_results(sheep_id: int,
                          frames_to_average: int = FRAMES_TO_AVERAGE,
                          method: str = AVERAGING_METHOD,
                          wapm_c: int = WAPM_C,
                          epsilon: float = EBWA_EPSILON) -> dict | None:
    """
    Run the full WTSA pipeline for one sheep experiment.

    Segmentation and DC computation are identical to the original Lambert pipeline.
    The only change is that np.mean is replaced with the selected weighted averager.

    Returns a dict containing:
      averaged_seg_array  : list length trim_sec; entry t is (100, n_ch) array or None
      amp                 : (trim_sec, n_ch) AC amplitude (max-min of weighted avg frame)
      dc_levels_per_frame : (trim_sec, n_ch) DC level from arithmetic mean of raw segments
      second_indices      : np.arange(trim_sec) — actual elapsed seconds, used as join key
      segment_start_indices, segment_raw_lengths : sample-level timing metadata
      segmented_hp        : accepted Segment objects (high-pass)
      ppg_cols, trim_sec, exp_id, exp_id_raw
    """
    exp_id_raw = get_sheep_experiment_id(sheep_id)["experiment_id"]
    exp_id = ensure_csv_suffix(exp_id_raw)
    sheep_df = get_sheep_dataframe(sheep_id).reset_index(drop=True)

    ac_cols = get_sorted_ac_columns()
    ppg_cols = ac_cols["wl1"] + ac_cols["wl2"]
    n_channels = len(ppg_cols)

    ppg_raw = sheep_df[ppg_cols].to_numpy(dtype=float)
    ppg_hp = np.apply_along_axis(lambda col: highpass_filter(col, FS), axis=0, arr=ppg_raw)

    fhr_1hz = sheep_df[["FHR"]].iloc[::FS].reset_index(drop=True)
    fhr_1hz = fhr_1hz.rolling(window=FHR_MA_WINDOW_SECONDS, min_periods=1, center=True).mean()
    fsao2_1hz = sheep_df["fSpO2"].iloc[::FS].reset_index(drop=True)

    ppg_hp, fhr_1hz, fsao2_1hz, trim_sec = _trim_to_common_length(ppg_hp, fhr_1hz, fsao2_1hz, FS)
    ppg_raw = ppg_raw[: int(ppg_hp.shape[0]), :]

    # Peak detection via frequency-warped channel 4 (WL1)
    ppg_ch4 = ppg_hp[:, 3]
    fhr_rep = np.repeat(fhr_1hz.values.flatten(), FS)[: len(ppg_ch4)]
    warped = fw.frequency_warpping_signal_extraction(ppg_ch4, FS, fhr_rep, 1)
    peaks = argrelextrema(warped, np.less)[0]

    # Segment both high-pass (for averaging) and raw (for DC)
    segmented_hp, _  = mypsa.signal_chopping(ppg_hp,  FS, peaks, fhr_1hz.values.flatten(), PEAK_DISTANCE_HINT)
    segmented_raw, _ = mypsa.signal_chopping(ppg_raw, FS, peaks, fhr_1hz.values.flatten(), PEAK_DISTANCE_HINT)
    segment_start_indices, segment_raw_lengths = _accepted_segment_meta(
        np.asarray(peaks, dtype=int), fhr_1hz.values.flatten()
    )

    n_segments = min(len(segmented_hp), len(segmented_raw),
                     len(segment_start_indices), len(segment_raw_lengths))
    segmented_hp          = segmented_hp[:n_segments]
    segmented_raw         = segmented_raw[:n_segments]
    segment_start_indices = segment_start_indices[:n_segments]
    segment_raw_lengths   = segment_raw_lengths[:n_segments]

    if n_segments == 0:
        return None

    # DC: arithmetic mean of raw segments (unchanged from original pipeline)
    _, dc_per_window = mypsa.compute_DC(segmented_raw, int(frames_to_average), trim_sec)
    dc_levels_per_frame = np.stack(
        [np.asarray(row, dtype=float).reshape(-1) for row in dc_per_window], axis=0
    )  # shape (trim_sec, n_channels)

    # Weighted averaging: iterate over every second t in [0, trim_sec)
    # averaged_seg_array[t] is None when no segments fall in window [t, t+frames_to_average)
    averaged_seg_array = []
    amp_list = []

    for t in range(trim_sec):
        window_segs = [seg for seg in segmented_hp
                       if t <= seg.Time < t + frames_to_average]
        if not window_segs:
            averaged_seg_array.append(None)
            amp_list.append(np.full(n_channels, np.nan))
            continue

        stack_3d = np.stack([np.asarray(seg.Array, dtype=float) for seg in window_segs], axis=0)
        if stack_3d.ndim == 2:
            stack_3d = stack_3d[:, :, None]

        N = stack_3d.shape[1]
        wtsa_frame = np.zeros((N, n_channels), dtype=float)
        for ch in range(n_channels):
            wtsa_frame[:, ch] = weighted_average_segments(
                stack_3d[:, :, ch], method=method, wapm_c=wapm_c, epsilon=epsilon
            )

        averaged_seg_array.append(wtsa_frame)
        amp_list.append(np.max(wtsa_frame, axis=0) - np.min(wtsa_frame, axis=0))

    return {
        "exp_id_raw"            : exp_id_raw,
        "exp_id"                : exp_id,
        "ppg_cols"              : ppg_cols,
        "trim_sec"              : trim_sec,
        "averaged_seg_array"    : averaged_seg_array,    # list[np.ndarray | None], len=trim_sec
        "amp"                   : np.stack(amp_list),    # (trim_sec, n_channels)
        "dc_levels_per_frame"   : dc_levels_per_frame,  # (trim_sec, n_channels)
        "segment_start_indices" : segment_start_indices,
        "segment_raw_lengths"   : segment_raw_lengths,
        "segmented_hp"          : segmented_hp,
    }


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
#  RUN PIPELINE ACROSS ALL SHEEP
# ═══════════════════════════════════════════════════════════════════════════

# Load ABG source once — used by both segment and EPR-only exports
abg_source = _load_abg_source()

all_results = {}
for sheep_idx in range(13):
    print(f"Processing sheep_idx={sheep_idx}...")
    result = compute_wtsa_results(sheep_idx)
    if result is None:
        print(f"  Skipped sheep_idx={sheep_idx} (no valid segments)")
    else:
        all_results[sheep_idx] = result

print(f"\nCompleted {len(all_results)}/13 sheep experiments.")


Processing sheep_idx=0...
Processing sheep_idx=1...
Processing sheep_idx=2...
Processing sheep_idx=3...
Processing sheep_idx=4...
Processing sheep_idx=5...
Processing sheep_idx=6...
Processing sheep_idx=7...
Processing sheep_idx=8...
Processing sheep_idx=9...
Processing sheep_idx=10...
Processing sheep_idx=11...
Processing sheep_idx=12...

Completed 13/13 sheep experiments.


In [7]:
# ═══════════════════════════════════════════════════════════════════════════
#  SEGMENT CSV EXPORT  (SAVE_SEGMENTS and/or SAVE_WITH_EPR)
# Schema matches ppg_frames.csv: one row per waveform sample
# ═══════════════════════════════════════════════════════════════════════════

if SAVE_SEGMENTS or SAVE_WITH_EPR:
    seg_records = []

    for sheep_idx, result in all_results.items():
        exp_id                 = result["exp_id"]
        exp_id_raw             = result["exp_id_raw"]
        ppg_cols               = result["ppg_cols"]
        trim_sec               = result["trim_sec"]
        segmented_hp           = result["segmented_hp"]
        segment_start_indices  = result["segment_start_indices"]
        segment_raw_lengths    = result["segment_raw_lengths"]
        dc_levels_per_frame    = result["dc_levels_per_frame"]
        averaged_seg_array     = result["averaged_seg_array"]
        n_channels             = len(ppg_cols)

        frame_number = 0
        for t in range(trim_sec):
            wtsa_frame = averaged_seg_array[t]
            if wtsa_frame is None:
                frame_number += 1
                continue

            # Determine precise time span from actual segment boundaries in this window
            seg_indices_in_window = [
                i for i, seg in enumerate(segmented_hp)
                if t <= seg.Time < t + FRAMES_TO_AVERAGE
            ]
            if not seg_indices_in_window:
                frame_number += 1
                continue

            first_abs = segment_start_indices[seg_indices_in_window[0]]
            last_abs  = (segment_start_indices[seg_indices_in_window[-1]]
                         + segment_raw_lengths[seg_indices_in_window[-1]])
            window_start_time = float(first_abs) / float(FS)
            window_end_time   = float(last_abs) / float(FS)

            N = wtsa_frame.shape[0]
            sample_time = (
                np.array([0.5 * (window_start_time + window_end_time)])
                if N == 1
                else np.linspace(window_start_time, window_end_time, N)
            )

            frame_df = pd.DataFrame(wtsa_frame, columns=ppg_cols)

            dc_row = np.full(n_channels, np.nan)
            if t < dc_levels_per_frame.shape[0]:
                dc_row = np.asarray(dc_levels_per_frame[t], dtype=float)
            for col_i, col_name in enumerate(ppg_cols):
                frame_df[f"dc_{col_name}"] = dc_row[col_i] if col_i < len(dc_row) else np.nan

            frame_df.insert(0, "time",                  sample_time)
            frame_df.insert(0, "sample_index_in_frame", np.arange(N, dtype=int))
            frame_df.insert(0, "frame_number",          int(frame_number))
            frame_df.insert(0, "experiment_id",         exp_id)
            seg_records.append(frame_df)
            frame_number += 1

    final_df = pd.concat(seg_records, ignore_index=True) if seg_records else pd.DataFrame()

    # ABG/FHR labels interpolated at each sample's absolute time
    if not final_df.empty:
        abg_parts = []
        for eid in final_df["experiment_id"].unique():
            mask  = final_df["experiment_id"] == eid
            times = final_df.loc[mask, "time"].to_numpy(dtype=float)
            abg_parts.append(_interp_abg_at_times(abg_source, eid, times))
        abg_all = pd.concat(abg_parts, ignore_index=True)
        final_df = pd.concat([final_df.reset_index(drop=True), abg_all.reset_index(drop=True)], axis=1)

    if SAVE_SEGMENTS and not final_df.empty:
        OUT_CSV_SEGMENTS.parent.mkdir(parents=True, exist_ok=True)
        final_df.to_csv(OUT_CSV_SEGMENTS, index=False)
        print(f"Saved {len(final_df)} segment rows to {OUT_CSV_SEGMENTS}")

    if SAVE_WITH_EPR and not final_df.empty:
        frame_groups = final_df.groupby(["experiment_id", "frame_number"])
        epr_headers = EPR_VALUE_HEADERS if len(EPR_VALUE_HEADERS) == 10 else                       [f"EPR{i+1} ({[15,30,45,70,100][i%5]}mm,{[740,850][i//5]}nm)" for i in range(10)]
        for i, col in enumerate(ORDERED_SIGNAL_COLS):
            if col in final_df.columns and f"dc_{col}" in final_df.columns:
                ac = frame_groups[col].transform("max") - frame_groups[col].transform("min")
                final_df[epr_headers[i]] = (ac + final_df[f"dc_{col}"]) / final_df[f"dc_{col}"]
        OUT_CSV_WITH_EPR.parent.mkdir(parents=True, exist_ok=True)
        final_df.to_csv(OUT_CSV_WITH_EPR, index=False)
        print(f"Saved {len(final_df)} rows to {OUT_CSV_WITH_EPR}")
else:
    final_df = pd.DataFrame()
    print("Skipping segment export (SAVE_SEGMENTS and SAVE_WITH_EPR both False).")


Skipping segment export (SAVE_SEGMENTS and SAVE_WITH_EPR both False).


In [8]:
# ═══════════════════════════════════════════════════════════════════════════
#  EPR-ONLY 1 Hz CSV EXPORT  (SAVE_EPR_ONLY)
#
#  Schema (matches lambert_lambert_combined.csv after randall_concat):
#    "" | seconds | EPR1..EPR10 | ABG/FHR columns | experiment_id
#
#  One row per second of recording.  "seconds" is the actual elapsed
#  recording second (t), used as the join key against fetal1_combined.csv
#  so ABG/FHR labels are correctly aligned even when some windows have no
#  valid segments.  Rows where no weighted average was computed (None frames)
#  still appear in the output with NaN EPR values so the 1 Hz grid is
#  unbroken — this matches what randall_concat produces after a left-join.
# ═══════════════════════════════════════════════════════════════════════════

if SAVE_EPR_ONLY:
    epr_headers = EPR_VALUE_HEADERS if len(EPR_VALUE_HEADERS) == 10 else                   [f"EPR{i+1} ({[15,30,45,70,100][i%5]}mm,{[740,850][i//5]}nm)" for i in range(10)]

    epr_rows = []
    skipped = []

    for sheep_idx, result in all_results.items():
        ppg_cols            = result["ppg_cols"]
        amp_arr             = result["amp"]               # (trim_sec, n_channels)
        dc_mat              = result["dc_levels_per_frame"]  # (trim_sec, n_channels)
        averaged_seg_array  = result["averaged_seg_array"]
        exp_id_raw          = result["exp_id_raw"]
        exp_id              = result["exp_id"]
        trim_sec            = result["trim_sec"]

        col_index = {name: ppg_cols.index(name)
                     for name in ORDERED_SIGNAL_COLS if name in ppg_cols}
        if len(col_index) != len(ORDERED_SIGNAL_COLS):
            skipped.append((sheep_idx, "missing expected ppg_cols"))
            continue

        # seconds = actual elapsed recording seconds (0..trim_sec-1)
        # This is the correct join key against fetal1_combined.csv
        second_values = np.arange(trim_sec, dtype=float)

        # Interpolate ABG/FHR labels at every second for this experiment
        abg_frame = _interp_abg_at_times(abg_source, exp_id_raw, second_values)

        for t in range(trim_sec):
            eprs = []
            for col in ORDERED_SIGNAL_COLS:
                ci = col_index[col]
                if averaged_seg_array[t] is None:
                    # Window had no valid segments — EPR is NaN but row still exists
                    eprs.append(float("nan"))
                else:
                    ac_hp   = float(amp_arr[t, ci])
                    dia_raw = float(dc_mat[t, ci])
                    eprs.append(epr_norm_ma(ac_hp, dia_raw))

            abg_vals = {col: float(abg_frame.iloc[t][col])
                        if col in abg_frame.columns else float("nan")
                        for col in ABG_COLUMNS}

            epr_rows.append({
                "row_index"    : len(epr_rows),
                "seconds"      : int(t),
                **{epr_headers[i]: eprs[i] for i in range(len(epr_headers))},
                **abg_vals,
                "experiment_id": exp_id,
            })

    if epr_rows:
        epr_df = pd.DataFrame(epr_rows)

        # Column order: "" (row index) | seconds | EPR cols | ABG cols | experiment_id
        col_order = ["row_index", "seconds", *epr_headers, *ABG_COLUMNS, "experiment_id"]
        col_order = [c for c in col_order if c in epr_df.columns]
        epr_df = epr_df[col_order].rename(columns={"row_index": ""})

        OUT_CSV_EPR_ONLY.parent.mkdir(parents=True, exist_ok=True)
        epr_df.to_csv(OUT_CSV_EPR_ONLY, index=False)
        print(f"Saved {len(epr_df)} rows to {OUT_CSV_EPR_ONLY}")
        print(f"  Columns: {epr_df.columns.tolist()}")
    else:
        print("No EPR rows produced.")

    if skipped:
        print("Skipped experiments:")
        for idx, err in skipped:
            print(f"  sheep_idx={idx}: {err}")
else:
    print("Skipping EPR-only export (SAVE_EPR_ONLY=False).")


Saved 32681 rows to /home/rlfowler/Documents/research/psafe_custom/data/wtsa_epr.csv
  Columns: ['', 'seconds', 'EPR1 (15mm,740nm)', 'EPR2 (30mm,740nm)', 'EPR3 (45mm,740nm)', 'EPR4 (70mm,740nm)', 'EPR5 (100mm,740nm)', 'EPR6 (15mm,850nm)', 'EPR7 (30mm,850nm)', 'EPR8 (45mm,850nm)', 'EPR9 (70mm,850nm)', 'EPR10 (100mm,850nm)', 'Fetal ABG pH', 'Fetal ABG sO2 (%)', 'Fetal ABG pO2 (mmHg)', 'Fetal ABG pCO2 (mmHg)', 'Fetal ABG BP', 'Fetal ABG Lac (mmol/L)', 'FHR_Clean', 'FHR', 'fSaO2', 'Fetal ABG HCO3 (mmol/L)', 'Fetal ABG BE (mmol/L)', 'Fetal ABG CBE (mmol/L)', 'Fetal ABG SBE (mmol/L)', 'Fetal ABG pH Change', 'Fetal ABG sO2 (%) Change', 'Fetal ABG pO2 (mmHg) Change', 'Fetal ABG pCO2 (mmHg) Change', 'Fetal ABG BP Change', 'Fetal ABG Lac (mmol/L) Change', 'FHR_Clean Change', 'FHR Change', 'fSaO2 Change', 'Fetal ABG HCO3 (mmol/L) Change', 'Fetal ABG BE (mmol/L) Change', 'Fetal ABG CBE (mmol/L) Change', 'Fetal ABG SBE (mmol/L) Change', 'experiment_id']


In [ ]:
import matplotlib.pyplot as plt

def plot_wtsa_frames(df: pd.DataFrame, experiment_id: str,
                     wl_channel: str = "ch4voltsWL1",
                     start_frame: int = 0, n_plot: int = 12):
    exp_df = df[df["experiment_id"] == experiment_id].copy()
    if exp_df.empty or wl_channel not in exp_df.columns:
        print(f"No data for experiment_id={experiment_id} / channel={wl_channel}")
        return
    frame_ids = np.sort(exp_df["frame_number"].dropna().unique().astype(int))
    selected = frame_ids[max(0, start_frame): max(0, start_frame) + int(n_plot)]
    ncols = 3
    nrows = int(np.ceil(len(selected) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 2.8 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for i, fid in enumerate(selected):
        fr = exp_df[exp_df["frame_number"] == int(fid)].sort_values("sample_index_in_frame")
        axes[i].plot(np.arange(len(fr)), fr[wl_channel].to_numpy(dtype=float),
                     linewidth=1.5, color="steelblue")
        axes[i].set_title(f"Frame {int(fid)}", fontsize=9)
        axes[i].grid(alpha=0.25)
    for j in range(len(selected), len(axes)):
        axes[j].axis("off")
    fig.suptitle(f"WTSA ({AVERAGING_METHOD}) | {experiment_id} | {wl_channel}", fontsize=12)
    fig.tight_layout()
    plt.show()

if not final_df.empty:
    _example_exp = final_df["experiment_id"].iloc[0]
    plot_wtsa_frames(final_df, experiment_id=_example_exp,
                     wl_channel="ch4voltsWL1", start_frame=0, n_plot=12)
else:
    print("No segment data to plot (SAVE_SEGMENTS and SAVE_WITH_EPR both False).")


No segment data to plot (SAVE_SEGMENTS and SAVE_WITH_EPR both False).


: 